# 34 · Distribuição de usuários por tipo de treino

Pergunta: como os usuários se distribuem por tipo de treino, nos três fatores (ativos, engajamento e churn)?

- **Tipo de treino de cada conta**: o tipo principal = o mais frequente nos 90 dias antes da data de
  referência (empate: o de mais minutos). Quem não treinou nos 90 dias fica em "sem treino". A tabela
  "pratica" conta a conta em todos os tipos que ela fez.
- **Ativos e engajamento**: assinantes ativos no corte (30/09/2026), com os níveis de `src/engajamento.py`.
- **Churn**: painel mensal de abr a set/2026 (os treinos começam em 01/01/2026 e o tipo principal
  precisa de 90 dias): tipo principal nos 90 dias antes do dia 1 de cada mês; churn = perdas ÷ ativas
  no início do mês.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
NIVEIS = ["1. alto", "2. médio", "3. baixo", "4. possível churn"]
TIPOS = ["corrida", "funcional", "caminhada", "forca", "yoga", "pedal"]

ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "duracao_min", "origem"])
assin = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "plano_atual", "inicio", "ativa_no_corte", "renovacao_automatica"])
usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "sexo", "idade_no_corte", "sistema"])
eng = pd.read_parquet(P / "engajamento_corte.parquet")
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "ativa_inicio_mes", "perdida_no_mes"])
print(len(ativ), "treinos |", len(assin), "assinaturas |", len(eng), "ativos com nível |", len(painel), "linhas do painel")

def perfil_de_treino(ref, dias=90):
    # tipo principal e variedade nos `dias` até `ref` (inclusive)
    a = ativ[(ativ.data <= ref) & (ativ.data > ref - pd.Timedelta(days=dias))]
    g = a.groupby(["user_id", "tipo"]).agg(n=("data", "size"), minutos=("duracao_min", "sum")).reset_index()
    g = g.sort_values(["user_id", "n", "minutos"], ascending=[True, False, False])
    top = g.drop_duplicates("user_id").set_index("user_id")
    tot = g.groupby("user_id").n.sum()
    r = pd.DataFrame({"principal": top.tipo, "treinos_90d": tot, "peso_principal": top.n / tot, "n_tipos": g.groupby("user_id").tipo.nunique()})
    pr = g.pivot(index="user_id", columns="tipo", values="n").fillna(0).reindex(columns=TIPOS, fill_value=0)
    return r.join(pr.add_prefix("n_"))

1252488 treinos | 32583 assinaturas | 14753 ativos com nível | 327771 linhas do painel


## 1. Ativos no corte por tipo de treino

In [2]:
at = assin[assin.ativa_no_corte].merge(usuarios, on="user_id", how="left").merge(eng[["user_id", "nivel", "treinos_30d"]], on="user_id", how="left", validate="one_to_one")
pf = perfil_de_treino(CORTE)
at = at.join(pf, on="user_id")
print("assinantes ativos:", len(at), "| com treino nos últimos 90 dias:", int(at.principal.notna().sum()), "| sem treino:", int(at.principal.isna().sum()))
at["principal"] = at.principal.fillna("sem treino")
for t in TIPOS: at["n_" + t] = at["n_" + t].fillna(0)

d = at.principal.value_counts().rename("ativos").to_frame()
d["% dos ativos"] = d.ativos / d.ativos.sum() * 100
d["% anual"] = at.groupby("principal").plano_atual.apply(lambda s: (s == "anual").mean() * 100)
d["treinos 90d (média)"] = at.groupby("principal").treinos_90d.mean()
d["peso do principal %"] = at.groupby("principal").peso_principal.mean() * 100
d["nº de tipos (média)"] = at.groupby("principal").n_tipos.mean()
print("== tipo PRINCIPAL dos assinantes ativos"); print(d); print()

pr = pd.DataFrame({"ativos que praticam": [(at["n_" + t] > 0).sum() for t in TIPOS]}, index=TIPOS)
pr["% dos ativos"] = pr["ativos que praticam"] / len(at) * 100
pr["% dos que praticam e têm como principal"] = [((at.principal == t).sum() / (at["n_" + t] > 0).sum()) * 100 for t in TIPOS]
pr["treinos 90d do tipo (média de quem pratica)"] = [at.loc[at["n_" + t] > 0, "n_" + t].mean() for t in TIPOS]
print("== tipos que os ativos PRATICAM (uma conta entra em vários)"); print(pr); print()
print("nº de tipos praticados em 90 dias"); x = at.n_tipos.fillna(0).astype(int).value_counts().sort_index().rename("ativos").to_frame(); x["%"] = x.ativos / x.ativos.sum() * 100; print(x)

assinantes ativos: 14753 | com treino nos últimos 90 dias: 13999 | sem treino: 754
== tipo PRINCIPAL dos assinantes ativos
            ativos  % dos ativos  % anual  treinos 90d (média)  peso do principal %  nº de tipos (média)
principal                                                                                               
corrida       5358          36.3     59.6                 26.5                 74.8                  3.7
funcional     3474          23.5     51.0                 29.7                 46.2                  3.8
caminhada     1777          12.0     50.4                 23.5                 68.5                  3.7
forca         1759          11.9     53.4                 24.6                 43.1                  3.6
pedal         1373           9.3     56.2                 27.7                 73.8                  3.9
sem treino     754           5.1     99.6                  NaN                  NaN                  NaN
yoga           258           1.7     

In [3]:
print("== tipo principal por plano (% da coluna)"); print(pd.crosstab(at.principal, at.plano_atual, normalize="columns") * 100); print()
print("== tipo principal por sexo (% da coluna)"); print(pd.crosstab(at.principal, at.sexo.fillna("não informado"), normalize="columns") * 100); print()
at["faixa"] = pd.cut(at.idade_no_corte.astype(float), [0, 34, 44, 54, 120], labels=["até 34", "35-44", "45-54", "55+"]).astype(object).fillna("não informado")
print("== tipo principal por faixa de idade (% da coluna)"); print(pd.crosstab(at.principal, at.faixa, normalize="columns") * 100); print()
print("== combinações: quem tem o tipo da linha como principal também pratica (% da linha)")
print((at[at.principal != "sem treino"].groupby("principal")[["n_" + t for t in TIPOS]].agg(lambda s: (s > 0).mean() * 100)).rename(columns=lambda c: c[2:]))

== tipo principal por plano (% da coluna)
plano_atual  anual  mensal
principal                 
caminhada     10.6    14.0
corrida       37.7    34.5
forca         11.1    13.1
funcional     20.9    27.1
pedal          9.1     9.6
sem treino     8.9     0.0
yoga           1.8     1.6

== tipo principal por sexo (% da coluna)
sexo          F    M  não informado
principal                          
caminhada  12.2 12.0           10.9
corrida    36.1 36.2           39.7
forca      11.6 12.3           11.4
funcional  23.3 23.9           22.4
pedal       9.9  8.8            8.0
sem treino  5.1  5.1            5.3
yoga        1.8  1.6            2.3

== tipo principal por faixa de idade (% da coluna)
faixa       35-44  45-54  55+  até 34  não informado
principal                                           
caminhada    12.1   12.0 12.2    11.5           12.6
corrida      36.7   36.0 35.7    37.5           36.5
forca        11.6   12.4 11.8    11.9           11.8
funcional    23.3   24.0 24.3   

## 2. Engajamento dos ativos por tipo principal

In [4]:
n = pd.crosstab(at.principal, at.nivel).reindex(columns=NIVEIS, fill_value=0)
e = (n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")).join(at.groupby("principal").treinos_30d.mean().rename("treinos 30d (média)"))
e["% renovação desligada"] = at.groupby("principal").renovacao_automatica.apply(lambda s: (s == "nao").mean() * 100)
e = e.sort_values("ativos", ascending=False); print(e); print()
for pl in ["mensal", "anual"]:
    k = at[at.plano_atual == pl]
    n = pd.crosstab(k.principal, k.nivel).reindex(columns=NIVEIS, fill_value=0)
    print(pl); print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")).sort_values("ativos", ascending=False)); print()
at["variedade"] = pd.cut(at.n_tipos.fillna(0), [-1, 0, 1, 2, 3, 6], labels=["sem treino", "1 tipo", "2 tipos", "3 tipos", "4+ tipos"])
n = pd.crosstab(at.variedade, at.nivel).reindex(columns=NIVEIS, fill_value=0)
print("por variedade de tipos"); print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")))

            1. alto  2. médio  3. baixo  4. possível churn  ativos  treinos 30d (média)  % renovação desligada
principal                                                                                                     
corrida        73.5      18.4       2.9                5.1    5358                 10.0                   17.2
funcional      81.6      14.3       1.7                2.3    3474                 11.1                    8.6
caminhada      74.5      18.0       3.4                4.2    1777                  8.9                   15.6
forca          77.1      17.1       2.4                3.4    1759                  9.3                   13.2
pedal          77.4      15.7       2.7                4.2    1373                 10.5                   13.5
sem treino      0.0       0.7       0.0               99.3     754                  0.0                   34.5
yoga           65.5      17.4       4.3               12.8     258                  5.5                   27.9



## 3. Churn por tipo principal (abr a set/2026)

In [5]:
partes = []
for mes in pd.date_range("2026-04-01", "2026-09-01", freq="MS"):
    k = painel[(painel.mes == mes) & painel.ativa_inicio_mes]
    pf = perfil_de_treino(mes - pd.Timedelta(days=1))
    k = k.join(pf[["principal", "n_tipos", "treinos_90d"]], on="user_id")
    print(mes.date(), "ativas no início do mês:", len(k), "| sem treino nos 90 dias:", int(k.principal.isna().sum()), "| perdas:", int(k.perdida_no_mes.sum()))
    partes.append(k)
pm = pd.concat(partes, ignore_index=True)
pm["principal"] = pm.principal.fillna("sem treino")
pm["variedade"] = pd.cut(pm.n_tipos.fillna(0), [-1, 0, 1, 2, 3, 6], labels=["sem treino", "1 tipo", "2 tipos", "3 tipos", "4+ tipos"])
print("assinatura-mês:", len(pm))

def churn(d, por):
    g = d.groupby(por, observed=True).agg(ativas=("user_id", "size"), perdas=("perdida_no_mes", "sum"))
    g["churn %"] = g.perdas / g.ativas * 100
    return g
print(); print("== todos os planos"); c0 = churn(pm, "principal").sort_values("ativas", ascending=False); c0["% das perdas"] = c0.perdas / c0.perdas.sum() * 100; c0["% da base"] = c0.ativas / c0.ativas.sum() * 100; print(c0)
cp = churn(pm, ["plano", "principal"])
print(); print("== churn % por plano"); print(cp["churn %"].unstack(0).join(cp.ativas.unstack(0), rsuffix=" (ativas)").loc[c0.index])
print(); print("== por variedade de tipos"); cv = churn(pm, ["plano", "variedade"]); print(cv["churn %"].unstack(0).join(cv.ativas.unstack(0), rsuffix=" (ativas)"))
print(); print("== mensal: churn % por mês e tipo principal"); print(churn(pm[pm.plano == "mensal"], ["mes", "principal"])["churn %"].unstack()[c0.index])

2026-04-01 ativas no início do mês: 14237 | sem treino nos 90 dias: 608 | perdas: 1154


2026-05-01 ativas no início do mês: 14119 | sem treino nos 90 dias: 677 | perdas: 1026


2026-06-01 ativas no início do mês: 14102 | sem treino nos 90 dias: 716 | perdas: 866
2026-07-01 ativas no início do mês: 14059 | sem treino nos 90 dias: 753 | perdas: 948


2026-08-01 ativas no início do mês: 14032 | sem treino nos 90 dias: 775 | perdas: 834
2026-09-01 ativas no início do mês: 14319 | sem treino nos 90 dias: 788 | perdas: 714
assinatura-mês: 84868

== todos os planos
            ativas  perdas  churn %  % das perdas  % da base
principal                                                   
corrida      31648    2290      7.2          41.3       37.3
funcional    18779     957      5.1          17.3       22.1
caminhada    10271     676      6.6          12.2       12.1
forca        10137     584      5.8          10.5       11.9
pedal         8200     569      6.9          10.3        9.7
sem treino    4317     350      8.1           6.3        5.1
yoga          1516     116      7.7           2.1        1.8



== churn % por plano
plano       anual  mensal  anual (ativas)  mensal (ativas)
principal                                                 
corrida       2.7    13.8           18754            12894
funcional     1.8     8.6            9668             9111
caminhada     3.5     9.8            5273             4998
forca         2.6     9.3            5290             4847
pedal         2.9    12.0            4587             3613
sem treino    8.0    17.9            4278               39
yoga          6.4     9.5             908              608

== por variedade de tipos
plano       anual  mensal  anual (ativas)  mensal (ativas)
variedade                                                 
sem treino    8.0    17.9            4278               39
1 tipo       10.8    18.2            4659              560
2 tipos       7.4    16.3            4048             1691
3 tipos       2.9    12.9            5764             4503
4+ tipos      0.8    10.4           30009            29317

== men

In [6]:
# Mesmo volume de treino: o tipo ainda importa? (mensal)
m = pm[(pm.plano == "mensal") & (pm.principal != "sem treino")].copy()
m["volume"] = pd.cut(m.treinos_90d, [0, 12, 30, 1000], labels=["1-12 treinos", "13-30", "31+"])
x = churn(m, ["volume", "principal"])
print("MENSAL: churn % por volume de treino em 90 dias e tipo principal"); print(x["churn %"].unstack(0)); print(x.ativas.unstack(0))
print()
# ar livre (corrida, caminhada, pedal) contra indoor, por sistema, em jun-jul (falha de GPS do Android)
u = pm.merge(usuarios[["user_id", "sistema"]], on="user_id", how="left")
u["ambiente"] = np.where(u.principal.isin(["corrida", "caminhada", "pedal"]), "ar livre", np.where(u.principal == "sem treino", "sem treino", "indoor"))
u["periodo"] = np.where(u.mes.isin(pd.to_datetime(["2026-06-01", "2026-07-01"])), "jun-jul", "abr-mai e ago-set")
print("MENSAL: churn % por ambiente do tipo principal, sistema e período")
print(churn(u[u.plano == "mensal"], ["periodo", "ambiente", "sistema"])["churn %"].unstack())

final = d.join(e[NIVEIS].add_prefix("% ")).join(c0[["churn %"]].rename(columns={"churn %": "churn mensal abr-set/26 %"})).join(cp["churn %"].unstack(0).add_prefix("churn % "))
final.to_csv(S / "tipo_treino_ativos_engajamento_churn.csv", encoding="utf-8-sig")
print(); print(final.T)

MENSAL: churn % por volume de treino em 90 dias e tipo principal
volume     1-12 treinos  13-30  31+
principal                          
caminhada          13.1    9.8  8.0
corrida            19.1   14.0 11.8
forca              12.5    8.8  8.2
funcional          12.5    8.5  7.7
pedal              15.0   11.6 11.4
yoga               11.4    8.5  8.2
volume     1-12 treinos  13-30   31+
principal                           
caminhada           927   2439  1632
corrida            1863   5729  5302
forca               859   2459  1529
funcional           986   4160  3965
pedal               513   1619  1481
yoga                228    307    73

MENSAL: churn % por ambiente do tipo principal, sistema e período
sistema                       android  ios
periodo           ambiente                
abr-mai e ago-set ar livre       12.4 11.7
                  indoor          9.8  8.8
                  sem treino     14.3 20.0
jun-jul           ar livre       18.4  9.3
                  indoor  